# 02 - SQL para Exploración de Datos

En esta etapa del proyecto se realiza la exploración de datos mediante consultas SQL 
sobre el dataset de TechVentas S.A.S. Para esto se carga el archivo CSV en una base 
de datos SQLite y se ejecutan 6 queries que permiten responder preguntas clave del 
negocio como productos disponibles, rendimiento por vendedor y cumplimiento de metas. 
Cada query se documenta en el archivo 02_sql_discovery.sql y se ejecuta desde este 
notebook usando pd.read_sql() para visualizar los resultados.

### 2.1 Carga del Dataset y Conexión a SQLite

In [1]:
import sqlite3
import pandas as pd

# Cargar el CSV
tabla1 = pd.read_csv('../data/ventas_techventas.csv', encoding='cp1252', parse_dates=['fecha'])
tabla1['vendedor'] = tabla1['vendedor'].str.replace('Ana Lï¿½ï¿', 'Ana López', regex=False) # Solución para un error en el desarrollo de Q2

# Conectar a SQLite y cargar la tabla
conn = sqlite3.connect('../output/techventas.db')
tabla1.to_sql('ventas', conn, if_exists='replace', index=False)

print("Tabla cargada correctamente")

Tabla cargada correctamente


### 2.2 Creación de Tabla Vendedores

In [2]:
conn.executescript("""
    -- ═══════════════════════════════════════════════════════
    -- TABLA: VENDEDORES
    -- Almacena la información de cada vendedor y su meta
    -- ═══════════════════════════════════════════════════════
    
    DROP TABLE IF EXISTS vendedores;  -- elimina la tabla si ya existe
    
    CREATE TABLE vendedores (
        id_vendedor   INTEGER PRIMARY KEY,
        vendedor      TEXT,
        zona          TEXT,
        meta_mensual  REAL
    );

    INSERT INTO vendedores VALUES (1, 'Ana López', 'Norte', 8000);
    INSERT INTO vendedores VALUES (2, 'Carlos Ruiz', 'Sur', 7500);
    INSERT INTO vendedores VALUES (3, 'Luis Mora', 'Norte', 8000);
    INSERT INTO vendedores VALUES (4, 'Maria Torres', 'Centro', 7000);
""")

print("Tabla vendedores creada correctamente")

Tabla vendedores creada correctamente


## 2.3 Consultas SQL - Data Discovery

### Q1 - Productos Únicos Disponibles


In [3]:
# Q1 - Productos únicos disponibles en el catálogo
q1 = pd.read_sql("""
    SELECT DISTINCT producto AS producto_disponible
    FROM ventas
    ORDER BY producto_disponible
""", conn)

print(q1)

  producto_disponible
0                 NaN
1       Laptop Pro 15
2   Mouse Inalambrico
3       Router WiFi 6
4             SSD 1TB
5    Teclado Mecanico


**Observación:** Se detectó 1 valor nulo en la columna producto. 
Este registro será eliminado en la etapa de limpieza con Pandas en la Parte 3.

### Q2 - Pedidos del Primer Trimestre


In [4]:
# Q2 - Pedidos del primer trimestre con cantidad mayor a 5
q2 = pd.read_sql("""
    SELECT *
    FROM ventas
    WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
    AND cantidad > 5
""", conn)

print(q2)

   order_id                fecha cliente_id     cliente_nombre  region  \
0      1002  2024-01-07 00:00:00       C002          Innovatek     Sur   
1      1004  2024-01-12 00:00:00       C001        TechCorp SA   Norte   
2      1006  2024-01-18 00:00:00       C005  Sistemas Globales   Oeste   
3      1008  2024-01-22 00:00:00       C006           NetPrime   Norte   
4      1009  2024-01-25 00:00:00       C003      DataSolutions  Centro   
5      1012  2024-02-05 00:00:00       C008           BetaSoft  Centro   
6      1015  2024-02-12 00:00:00       C005  Sistemas Globales   Oeste   
7      1018  2024-02-20 00:00:00       C002          Innovatek     Sur   
8      1019  2024-02-22 00:00:00       C007          AlphaTech     Sur   
9      1022  2024-03-04 00:00:00       C001        TechCorp SA   Norte   
10     1023  2024-03-07 00:00:00       C009          GammaDevs   Oeste   
11     1026  2024-03-15 00:00:00       C005  Sistemas Globales   Oeste   
12     1027  2024-03-18 00:00:00      

**Observación:** Se encontraron 15 pedidos en el primer trimestre (ene-mar 2024) 
con cantidad mayor a 5 unidades. Se detectó un nombre de vendedor con caracteres 
especiales ('Ana Ló...') debido al encoding del archivo original.

Decidi buscar una manera de corregir este nombre asi que la puse en las primeras lineas de codigo, lo anoto aca porque el error ocurrio en este punto. (el nombre aparecia como Ana Lï¿½ï¿)

### Q3 - Vendedores con Mayor Ingreso


In [5]:
# Q3 - Vendedores con ingreso bruto total mayor a $10,000
q3 = pd.read_sql("""
    SELECT vendedor, 
           SUM(cantidad * precio_unitario) AS ingreso_bruto_total
    FROM ventas
    GROUP BY vendedor
    HAVING ingreso_bruto_total > 10000
    ORDER BY ingreso_bruto_total DESC
""", conn)

print(q3)

       vendedor  ingreso_bruto_total
0   Carlos Ruiz              21355.0
1     Ana López              20810.0
2     Luis Mora              19830.0
3  Maria Torres              11860.0


**Observación:** Los 4 vendedores superan el umbral de $10,000 en ingreso bruto total.
Carlos Ruiz lidera con $21,355 seguido de Ana López con $20,810. 
Maria Torres es la de menor ingreso pero aún así supera el umbral con $11,860.

### Q4 - Resumen por Categoría


In [6]:
# Q4 - Resumen por categoría
q4 = pd.read_sql("""
    SELECT categoria,
           COUNT(order_id) AS total_pedidos,
           SUM(cantidad) AS total_unidades,
           ROUND(AVG(precio_unitario), 2) AS precio_promedio
    FROM ventas
    GROUP BY categoria
    ORDER BY total_pedidos DESC
""", conn)

print(q4)

        categoria  total_pedidos  total_unidades  precio_promedio
0     Perifericos             15           215.0            53.00
1     Electronica             14            31.0          1139.29
2  Almacenamiento             12           260.0            95.00
3             NaN             11             NaN              NaN
4           Redes              8            39.0           120.00


**Observación:** Se identificaron 4 categorías: Periféricos, Electrónica, 
Almacenamiento y Redes. Se detectó 1 registro con categoría nula que será 
tratado en la limpieza con Pandas en la Parte 3.

Almacenamiento tiene el mayor volumen de unidades vendidas (260) mientras que 
Electrónica tiene el precio promedio más alto ($1,139.29), lo que sugiere 
productos premium en esa categoría.

### Q5 - Cumplimiento de Meta por Vendedor


In [7]:
# Q5 - Cumplimiento de meta por vendedor
q5 = pd.read_sql("""
    SELECT v.vendedor,
           v.zona,
           v.meta_mensual,
           SUM(ve.cantidad * ve.precio_unitario) AS ingreso_total,
           ROUND(SUM(ve.cantidad * ve.precio_unitario) / (v.meta_mensual * 6) * 100, 2) AS pct_cumplimiento
    FROM vendedores v
    JOIN ventas ve ON v.vendedor = ve.vendedor
    GROUP BY v.vendedor
    ORDER BY pct_cumplimiento DESC
""", conn)

print(q5)

       vendedor    zona  meta_mensual  ingreso_total  pct_cumplimiento
0   Carlos Ruiz     Sur        7500.0        21355.0             47.46
1     Ana López   Norte        8000.0        20810.0             43.35
2     Luis Mora   Norte        8000.0        19830.0             41.31
3  Maria Torres  Centro        7000.0        11860.0             28.24


**Observación:** Carlos Ruiz lidera el cumplimiento de meta con 47.46%, 
seguido de Ana López con 43.35%. Sin embargo ningún vendedor supera el 
50% de cumplimiento del semestre, lo que podría ser una señal de alerta 
para el CEO sobre el rendimiento general del equipo de ventas.

### Q6 - Cliente con Mayor Ingreso Total


In [8]:
# Q6 - Cliente con mayor ingreso total en el primer semestre
q6 = pd.read_sql("""
    SELECT cliente_id,
           cliente_nombre,
           SUM(cantidad * precio_unitario) AS ingreso_total
    FROM ventas
    GROUP BY cliente_id, cliente_nombre
    HAVING ingreso_total = (
        SELECT MAX(ingreso_total)
        FROM (
            SELECT SUM(cantidad * precio_unitario) AS ingreso_total
            FROM ventas
            GROUP BY cliente_id
        )
    )
""", conn)

print(q6)

  cliente_id cliente_nombre  ingreso_total
0       C001    TechCorp SA        19025.0


**Observación:** El cliente con mayor ingreso total en el primer semestre 
es TechCorp SA (C001) con $19,025. Este cliente es estratégico para 
TechVentas y merece atención especial para mantener la relación comercial.

In [9]:
# Cerrar conexión a la base de datos
conn.close()
print("Conexión cerrada")

Conexión cerrada
